In [ ]:
from pathlib import Path
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np

# -----------------------------
# Path to your unzipped data
# -----------------------------
base_dir = Path("../data/visium_zip/11a4e71d2bb02_VisiumHD_HumanColon_Oliveira")
bin_dir = base_dir / "binned_outputs" / "square_016um"

h5_path = bin_dir / "filtered_feature_bc_matrix.h5"
pos_path = bin_dir / "spatial" / "tissue_positions.parquet"
cluster_path = bin_dir / "clustering.csv.gz"


# -------------------------------------------------
# Paths
# -------------------------------------------------
pos_path = bin_dir / "spatial" / "tissue_positions.parquet"
cluster_path = bin_dir / "clustering.csv.gz"

assert pos_path.exists(), f"Missing spatial positions file: {pos_path}"

# -------------------------------------------------
# Read expression matrix
# -------------------------------------------------
adata = sc.read_10x_h5(h5_path)
adata.var_names_make_unique()

print("Expression matrix before matching:", adata.shape)

# -------------------------------------------------
# Read spatial coordinates
# -------------------------------------------------
positions = pd.read_parquet(pos_path)

if "barcode" in positions.columns:
    positions = positions.set_index("barcode")

# -------------------------------------------------
# Match shared barcodes only
# -------------------------------------------------
common_barcodes = adata.obs_names.intersection(positions.index)

print("Expression barcodes:", adata.n_obs)
print("Position barcodes:", positions.shape[0])
print("Matched barcodes:", len(common_barcodes))

if len(common_barcodes) == 0:
    raise ValueError("No matching barcodes found.")

adata = adata[common_barcodes].copy()
positions = positions.loc[common_barcodes]

print("Expression matrix after matching:", adata.shape)

# -------------------------------------------------
# Add spatial metadata
# -------------------------------------------------
adata.obs = adata.obs.join(positions)

coord_cols = ["pxl_col_in_fullres", "pxl_row_in_fullres"]

missing_cols = [c for c in coord_cols if c not in adata.obs.columns]
if missing_cols:
    raise KeyError(f"Missing coordinate columns: {missing_cols}")

adata.obsm["spatial"] = adata.obs[coord_cols].to_numpy()

# -------------------------------------------------
# Add precomputed clustering safely
# -------------------------------------------------
if cluster_path.exists():
    clusters = pd.read_csv(cluster_path)

    print("Cluster columns:")
    print(clusters.columns.tolist())

    if "barcode" in clusters.columns:
        clusters = clusters.set_index("barcode")

        common_cluster_barcodes = adata.obs_names.intersection(clusters.index)

        for col in clusters.columns:
            new_col = f"precomputed_{col}"

            # Start as missing strings, not pd.NA mixed objects
            adata.obs[new_col] = "missing"

            # Fill matched values as strings
            adata.obs.loc[common_cluster_barcodes, new_col] = (
                clusters.loc[common_cluster_barcodes, col]
                .astype(str)
                .values
            )

            # Convert to categorical to save cleanly in h5ad
            adata.obs[new_col] = adata.obs[new_col].astype("category")

# -------------------------------------------------
# Basic QC
# -------------------------------------------------
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs["n_genes_by_counts"] = np.asarray((adata.X > 0).sum(axis=1)).ravel()

# -------------------------------------------------
# Clean obs columns before saving
# This prevents mixed object dtype errors
# -------------------------------------------------
for col in adata.obs.columns:
    if adata.obs[col].dtype == "object":
        adata.obs[col] = adata.obs[col].astype(str)

# -------------------------------------------------
# Save AnnData
# -------------------------------------------------
out_path = "../data/visiumhd_human_colon_16um.h5ad"
adata.write_h5ad(out_path)

print("\nFinal AnnData:")
print(adata)

print("\nSaved to:")
print(out_path)